# CIFAR-10 Architectural Benchmark: KasraNet-V1-IncepNeXt vs. Classical & Multi-Branch CNNs

[![PyTorch](https://img.shields.io/badge/PyTorch-2.x-EE4C2C?logo=pytorch&logoColor=white)](https://pytorch.org/)
[![Repository](https://img.shields.io/badge/GitHub-vision--architectures--pytorch-181717?logo=github)](https://github.com/mohammadrezakasra/vision-architectures-pytorch)
[![Dataset](https://img.shields.io/badge/Dataset-CIFAR--10-blue)](https://www.cs.toronto.edu/~kriz/cifar.html)
[![Precision](https://img.shields.io/badge/Precision-FP16%20AMP%20%7C%20channels__last-success)]()

> **Language:** [English](#1-overview) | [فارسی](#بخش-فارسی-بنچمارک-جامع-معماریهای-بینایی-ماشین-روی-cifar-10)

---

## 1. Overview

This notebook presents a controlled, end-to-end empirical benchmark evaluating four convolutional neural network architectures on **CIFAR-10** under identical data splits, augmentation pipelines, hyperparameters, and mixed-precision (`AMP`) optimization schedules. Alongside three established and lightweight baselines (**WideResNet-28-2**, **ResNeXt-29**, and **DualBranchResNet-28**), we introduce and evaluate **`KasraNet-V1-IncepNeXt`**—a custom hybrid architecture combining multi-scale spatial feature extraction (`1×1`, `3×3`, `5×5`) with grouped convolutions. Beyond standard classification accuracy, this study profiles theoretical complexity (**Parameters, MACs, FLOPs**) against real-world hardware execution speed (**GPU Batch Latency & Throughput**).

---

## 2. Evaluated Architectures

| Architecture | Design Paradigm | Key Configuration |
| :--- | :--- | :--- |
| **`KasraNet-V1-IncepNeXt`** *(Custom)* | Multi-Scale InceptionNeXt + Grouped Convolutions | `layers=28`, `cardinality=16`, `wide_factor=2` |
| **`ResNeXt-29`** | Aggregated Residual Transformations (Grouped Bottleneck) | `depth=29`, `cardinality=2` |
| **`WideResNet-28-2`** | Wide Residual Network (Dense $3\times3$ Convolutions) | `depth=28`, `widen_factor=2` |
| **`DualBranchResNet-28`** | Lightweight Parallel Two-Branch Residual Network | `depth=28`, Dual-Path Feature Split |

---

## 3. Experimental Setup & Training Pipeline

### A. Dataset & Augmentation
* **Dataset Split:** Stratified 90/10 split of the CIFAR-10 training set (`45,000` Training / `5,000` Validation / `10,000` Unseen Test images, `random_state=42`).
* **Training Augmentations:** `RandomCrop(32, padding=4)` $\rightarrow$ `RandomHorizontalFlip(p=0.5)` $\rightarrow$ `RandomErasing(p=0.5, scale=(0.02, 0.25))` $\rightarrow$ Channel Normalization.
* **Evaluation Transform:** Clean `ToTensor()` + Channel Normalization (no test-time augmentation).

### B. Optimization & Hardware Acceleration
* **Optimizer & Loss:** `SGD(lr=0.1, momentum=0.9, weight_decay=5e-4)` with `CrossEntropyLoss(label_smoothing=0.1)`.
* **LR Schedule & Epochs:** `CosineAnnealingLR(T_max=150, eta_min=1e-6)` over `150` epochs (`batch_size=256`).
* **System & GPU Optimizations:** Executed on **NVIDIA Tesla T4** using `torch.amp.autocast('cuda')` (FP16 Mixed Precision), `torch.channels_last` tensor memory layout, `torch.backends.cudnn.benchmark = True`, `num_workers=2`, and `pin_memory=True`.

---

## 4. Final Benchmark Results

> **Profiling Methodology:** Theoretical complexity (`Params`, `MACs`, `FLOPs`) was profiled via `thop` on a $1 \times 3 \times 32 \times 32$ tensor. Final accuracy and GPU latency (`Batch Size = 256`, 30 warmup runs + 100 timed runs via `torch.cuda.Event`) were evaluated on clean model instances without profiling hooks.

| Rank (Acc) | Model | Params | MACs | FLOPs | Best Val Acc | Best Val Loss | **Test Acc** | **Test Loss** | Batch Latency (`BS=256`) | Throughput (`img/s`) |
| :---: | :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| **1** | **`KasraNet-V1-IncepNeXt`** | 1.06M | 157.67M | 315.33M | **95.66%** | **0.6112** | **95.27%** | **0.6166** | 71.60 ms | 3,575 |
| **2** | **`ResNeXt-29`** | 1.42M | 216.88M | 433.77M | 95.36% | 0.6274 | **95.15%** | 0.6324 | 22.32 ms | 11,467 |
| **3** | **`WideResNet-28-2`** | 1.47M | 216.26M | 432.52M | 94.72% | 0.6325 | **94.45%** | 0.6397 | 13.56 ms | 18,879 |
| **4** | **`DualBranchResNet-28`** | **0.74M** | **111.19M** | **222.37M** | 94.86% | 0.6334 | **94.34%** | 0.6399 | **12.40 ms** | **20,645** |

---

## 5. Key Engineering Insights & Analysis

1. **State-of-the-Art Accuracy & Parameter Efficiency (`KasraNet-V1-IncepNeXt`):**
   `KasraNet-V1-IncepNeXt` achieves the highest validation (**95.66%**) and test accuracy (**95.27%**) alongside the lowest test cross-entropy loss (**0.6166**), outperforming `WideResNet-28-2` by **+0.82%** and `ResNeXt-29` by **+0.12%** on the test set while requiring **~25% fewer parameters** (`1.06M` vs. `1.42M`) and **~27% fewer FLOPs** (`315.33M` vs. `433.77M`). Multi-scale receptive fields (`1×1`, `3×3`, `5×5`) combined with grouped convolutions yield superior feature calibration on low-resolution $32\times32$ inputs.
2. **The FLOPs vs. Wall-Clock Latency Paradox:**
   Despite consuming 27% fewer FLOPs than `WideResNet-28-2`, `KasraNet-V1-IncepNeXt` exhibits higher GPU batch latency (`71.60 ms` vs. `13.56 ms`). Consistent with findings in *ShuffleNetV2*, *RepVGG*, and *InceptionNeXt*, this stems from two hardware bottlenecks:
   * **Network Fragmentation & Kernel Launch Overhead:** Parallel multi-scale branches (`1×1`, `3×3`, `5×5`) and tensor concatenations (`torch.cat`) force the GPU to dispatch numerous small CUDA kernels sequentially rather than saturating Tensor Cores with large, monolithic matrix multiplications.
   * **Memory-Bandwidth Bound Execution (High MAC):** High-cardinality grouped convolutions (`cardinality=16`) have lower arithmetic intensity (compute-to-memory-access ratio) than dense $3\times3$ convolutions, making inference bottlenecked by VRAM read/write bandwidth.
3. **Real-Time Throughput Winner (`DualBranchResNet-28`):**
   With only **0.74M parameters** and **222.37M FLOPs**, `DualBranchResNet-28` achieves **94.34%** test accuracy (nearly matching `WideResNet-28-2` at half the compute budget) while delivering the fastest inference speed at **12.40 ms/batch** (**20,645 images/sec**).
4. **Future Direction (`KasraNet-V2`):**
   To bridge the gap between theoretical FLOPs and GPU wall-clock latency, `KasraNet-V2` will integrate **Structural Re-parameterization** (fusing parallel multi-scale branches into a single unified convolution at inference time) alongside `torch.compile` / TensorRT graph fusion.

---

## 6. Notebook Execution Flow

1. **Repository & Environment Setup:** Cloning `vision-architectures-pytorch` and configuring PyTorch/cuDNN flags.
2. **Dataset & Augmentation Pipeline:** Preparing stratified CIFAR-10 splits (`45k / 5k / 10k`) and `DataLoader` workers.
3. **Model Instantiation & Training:** Training all four architectures for 150 epochs with `AMP` and `channels_last`, saving best checkpoints to `./checkpoints/`.
4. **Complexity Profiling (`thop`):** Measuring exact Parameters, MACs, and FLOPs for each architecture.
5. **Clean Test Evaluation & GPU Latency Benchmarking:** Loading clean checkpoints without profiling hooks to evaluate final test accuracy, test loss, batch latency (`ms`), and throughput (`img/s`).

---
---

<div dir="rtl">

# بخش فارسی: بنچمارک جامع معماری‌های بینایی ماشین روی CIFAR-10

## ۱. معرفی پروژه

این نوت‌بوک یک بنچمارک تجربی و کاملاً کنترل‌شده برای ارزیابی چهار معماری شبکه عصبی کانولوشنی روی دیتاست **CIFAR-10** است که در آن تمامی مدل‌ها تحت تقسیم‌بندی داده، پایپ‌لاین افزونه‌سازی داده (Augmentation)، هایپرپارامترها و زمان‌بندی بهینه‌سازی کاملاً یکسان (`AMP`) آموزش دیده‌اند. در کنار سه معماری مرجع و سبک (**WideResNet-28-2**، **ResNeXt-29** و **DualBranchResNet-28**)، معماری اختصاصی **`KasraNet-V1-IncepNeXt`** که ترکیبی از استخراج ویژگی چندمقیاسی (`1×1`، `3×3`، `5×5`) و کانولوشن‌های گروهی است معرفی و ارزیابی شده است. در این مطالعه، علاوه بر دقت نهایی دسته‌بندی، پیچیدگی تئوری (**تعداد پارامترها، MACs و FLOPs**) در برابر سرعت واقعی اجرا روی سخت‌افزار (**تأخیر بچ و نرخ پردازش بر ثانیه روی GPU**) سنجیده شده است.

---

## ۲. معماری‌های مورد بررسی

| نام معماری | الگوی طراحی | پیکربندی کلیدی |
| :--- | :--- | :--- |
| **`KasraNet-V1-IncepNeXt`** *(معماری پیشنهادی)* | بلوک‌های چندمقیاسی InceptionNeXt + کانولوشن گروهی | `layers=28`, `cardinality=16`, `wide_factor=2` |
| **`ResNeXt-29`** | تبدیل‌های باقیمانده تجمیعی (Grouped Bottleneck) | `depth=29`, `cardinality=2` |
| **`WideResNet-28-2`** | شبکه باقیمانده عریض (کانولوشن‌های متراکم $3\times3$) | `depth=28`, `widen_factor=2` |
| **`DualBranchResNet-28`** | شبکه باقیمانده سبک‌وزن دو شاخه‌ای موازی | `depth=28`, تقسیم مسیر دوگانه ویژگی‌ها |

---

## ۳. پیکربندی آزمایش و خط لوله آموزش

### الف) دیتاست و افزونه‌سازی داده (Augmentation)
* **تقسیم‌بندی داده:** جداسازی لایه‌ای (Stratified) با نسبت ۹۰ به ۱۰ روی مجموعه آموزش CIFAR-10 شامل **۴۵,۰۰۰ تصویر آموزش**، **۵,۰۰۰ تصویر اعتبارسنجی** و **۱۰,۰۰۰ تصویر تست کاملاً دیده‌نشده** (`random_state=42`).
* **تکنیک‌های آموزش:** اعمال ترتیب `RandomCrop(32, padding=4)` $\leftarrow$ `RandomHorizontalFlip(p=0.5)` $\leftarrow$ `RandomErasing(p=0.5, scale=(0.02, 0.25))` $\leftarrow$ نرمال‌سازی کانال‌ها.
* **تبدیل ارزیابی:** تبدیل استاندارد `ToTensor()` همراه با نرمال‌سازی (بدون اعمال Test-Time Augmentation).

### ب) بهینه‌سازی و شتاب‌دهی سخت‌افزاری
* **الگوریتم بهینه‌ساز و تابع خطا:** بهینه‌ساز `SGD(lr=0.1, momentum=0.9, weight_decay=5e-4)` همراه با `CrossEntropyLoss(label_smoothing=0.1)`.
* **زمان‌بندی نرخ یادگیری:** اسکجولر کسینوسی `CosineAnnealingLR(T_max=150, eta_min=1e-6)` در طول **۱۵۰ ایپاک** با اندازه بچ `256`.
* **بهینه‌سازی سخت‌افزاری:** اجرا روی پردازنده گرافیکی **NVIDIA Tesla T4** با فعال‌سازی دقت ترکیبی خودکار (`AMP`)، چیدمان حافظه `torch.channels_last` برای اشباع هسته‌های Tensor Core و تنظیمات `cudnn.benchmark = True` و `pin_memory=True`.

---

## ۴. نتایج نهایی بنچمارک

> **نحوه اندازه‌گیری:** پیچیدگی تئوری (`Params`، `MACs` و `FLOPs`) با کتابخانه `thop` روی تانسور ورودی $1 \times 3 \times 32 \times 32$ محاسبه شده است. دقت نهایی و تأخیر واقعی GPU (`Batch Size = 256`، شامل ۳۰ اجرای گرم‌کردن اولیه + ۱۰۰ اجرای زمان‌سنجی‌شده با `torch.cuda.Event`) روی نمونه‌های تمیز مدل‌ها و بدون هوک‌های پروفایلینگ ثبت شده‌اند.

| رتبه دقت | نام مدل | پارامترها | MACs | FLOPs | بهترین Val Acc | بهترین Val Loss | **دقت تست (Test Acc)** | **خطای تست (Test Loss)** | تأخیر بچ (`BS=256`) | نرخ پردازش (`img/s`) |
| :---: | :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| **۱** | **`KasraNet-V1-IncepNeXt`** | 1.06M | 157.67M | 315.33M | **95.66%** | **0.6112** | **95.27%** | **0.6166** | 71.60 ms | 3,575 |
| **۲** | **`ResNeXt-29`** | 1.42M | 216.88M | 433.77M | 95.36% | 0.6274 | **95.15%** | 0.6324 | 22.32 ms | 11,467 |
| **۳** | **`WideResNet-28-2`** | 1.47M | 216.26M | 432.52M | 94.72% | 0.6325 | **94.45%** | 0.6397 | 13.56 ms | 18,879 |
| **۴** | **`DualBranchResNet-28`** | **0.74M** | **111.19M** | **222.37M** | 94.86% | 0.6334 | **94.34%** | 0.6399 | **12.40 ms** | **20,645** |

---

## ۵. تحلیل مهندسی و یافته‌های کلیدی

1. **قهرمان دقت، کالیبراسیون و بهره‌وری پارامتری (`KasraNet-V1-IncepNeXt`):**
   معماری `KasraNet-V1-IncepNeXt` با ثبت بالاترین دقت اعتبارسنجی (**۹۵.۶۶٪**) و دقت تست (**۹۵.۲۷٪**) همراه با کمترین خطای تست (**۰.۶۱۶۶**)، از `WideResNet-28-2` به اندازه **+۰.۸۲٪** و از `ResNeXt-29` به اندازه **+۰.۱۲٪** جلوتر ایستاد؛ آن هم در حالی که حدود **۲۵٪ پارامتر کمتر** (`1.06M` در برابر `1.42M`) و **۲۷٪ عملیات محاسباتی کمتر** (`315.33M` در برابر `433.77M` FLOPs) مصرف کرده است. ترکیب میدان‌های دید چندمقیاسی (`1×1`، `3×3` و `5×5`) با کانولوشن‌های گروهی، مرزهای تصمیم‌گیری بسیار دقیق‌تر و تعمیم‌پذیرتری روی تصاویر کم‌رزولوشن $32\times32$ ایجاد کرده است.

2. **پارادوکس FLOPs تئوری در برابر تأخیر واقعی GPU:**
   با وجود ۲۷٪ محاسبات کمتر نسبت به `WideResNet-28-2`، زمان اجرای هر بچ در `KasraNet-V1` بیشتر شده است (`71.60 ms` در برابر `13.56 ms`). این پدیده که در مقالات مرجع معماری‌های بهینه (مانند *ShuffleNetV2*، *RepVGG* و *InceptionNeXt*) نیز بررسی شده، دو دلیل سخت‌افزاری دارد:
   * **تکه‌تکه شدن شبکه (Fragmentation) و سربار فراخوانی کرنل:** وجود شاخه‌های موازی چندمقیاسی و الحاق مداوم تانسورها (`torch.cat`) باعث می‌شود GPU به جای اجرای یک ضرب ماتریسی یکپارچه، ده‌ها کرنل کوچک CUDA را پشت سر هم اجرا کند.
   * **محدودیت پهنای باند حافظه (Memory-Bandwidth Bound):** کانولوشن‌های گروهی با کاردینالیتی بالا (`cardinality=16`) نسبت محاسبه به انتقال حافظه (Arithmetic Intensity) پایین‌تری دارند؛ بنابراین هسته‌های Tensor Core در `WideResNet` با حداکثر توان اشباع می‌شوند، اما در معماری چندشاخه‌ای زمان بیشتری صرف خواندن و نوشتن تانسورها در حافظه VRAM می‌شود.

3. **پادشاه سرعت و استنتاج بلادرنگ (`DualBranchResNet-28`):**
   این مدل با تنها **۰.۷۴ میلیون پارامتر** و نصف عملیات محاسباتی مدل‌های مرجع، دقت **۹۴.۳۴٪** (هم‌تراز با `WideResNet-28-2`) را با سریع‌ترین زمان پاسخ‌گویی یعنی **۱۲.۴۰ میلی‌ثانیه به ازای هر بچ** (**۲۰,۶۴۵ تصویر در ثانیه**) ارائه می‌دهد.

4. **نقشه راه نسخه بعدی (`KasraNet-V2`):**
   برای حفظ برتری دقت چندمقیاسی و رساندن سرعت استنتاج GPU به محدوده زیر ۲۰ میلی‌ثانیه، در نسخه `KasraNet-V2` از تکنیک **بازپارامتردهی ساختاری (Structural Re-parameterization)** برای ادغام شاخه‌های موازی در زمان `eval()` و بهینه‌سازی گراف با `torch.compile` / TensorRT استفاده خواهد شد.

---

## ۶. ساختار اجرایی نوت‌بوک

1. **تنظیم محیط و مخزن:** کلون کردن مخزن `vision-architectures-pytorch` و پیکربندی پرچم‌های بهینه‌سازی PyTorch/cuDNN.
2. **آماده‌سازی دیتاست:** تقسیم لایه‌ای CIFAR-10 (`45k / 5k / 10k`) و ساخت `DataLoader`ها.
3. **ساخت و آموزش مدل‌ها:** آموزش هر چهار معماری به مدت ۱۵۰ ایپاک با `AMP` و `channels_last` و ذخیره بهترین وزن‌ها در `./checkpoints/`.
4. **پروفایلینگ پیچیدگی محاسباتی (`thop`):** محاسبه دقیق پارامترها، MACs و FLOPs هر مدل.
5. **ارزیابی نهایی تست و زمان‌سنجی GPU:** بارگذاری وزن‌های تمیز بدون هوک و اندازه‌گیری دقت تست، خطای تست، تأخیر بچ (`ms`) و نرخ پردازش (`img/s`).

</div>

In [1]:
!git clone 'https://github.com/mohammadrezakasra/vision-architectures-pytorch.git'

Cloning into 'vision-architectures-pytorch'...
remote: Enumerating objects: 164, done.
remote: Counting objects: 100% (164/164), done.
remote: Compressing objects: 100% (122/122), done.
remote: Total 164 (delta 90), reused 103 (delta 39), pack-reused 0 (from 0)
Receiving objects: 100% (164/164), 67.25 KiB | 2.40 MiB/s, done.
Resolving deltas: 100% (90/90), done.


In [2]:
import sys
sys.path.append('/kaggle/working/vision-architectures-pytorch')

In [3]:
import torch
import torch.nn as nn
import torchvision
from torch.utils import data
from torchvision import transforms
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader,Subset

from models.cnn import wide_resnet_28, dual_branch_resnet_28
from utils.trainer import Trainer
from models.cnn.dual_branch_resnet_28 import DualBranchResNet28
from models.cnn.resnext_29 import ResNext29
from models.cnn.wide_resnet_28 import WideResNet28
from models.cnn.KasraNet_V1_IncepNeXt import KasraNet_V1_IncepNeXt
import sys
import os

In [16]:
torch.backends.cudnn.benchmark = True

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

## download Data & transform

In [5]:
!sudo apt-get install -y aria2

!aria2c -x 16 -s 16 https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz -d ./data/

!tar -xvf ./data/cifar-10-python.tar.gz -C ./data/

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libaria2-0 libcares2
The following NEW packages will be installed:
  aria2 libaria2-0 libcares2
0 upgraded, 3 newly installed, 0 to remove and 54 not upgraded.
Need to get 1,566 kB of archives.
After this operation, 5,692 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 libcares2 amd64 1.27.0-1.0ubuntu1.1 [74.2 kB]
Get:2 http://archive.ubuntu.com/ubuntu noble/universe amd64 libaria2-0 amd64 1.37.0+debian-1build3 [1,105 kB]
Get:3 http://archive.ubuntu.com/ubuntu noble/universe amd64 aria2 amd64 1.37.0+debian-1build3 [387 kB]
Fetched 1,566 kB in 0s (4,381 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 3.)
debcon

In [21]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomCrop(32 ,padding= 4),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.5, scale=(0.02, 0.25), value="random"),
    transforms.Normalize((.5,.5,.5),(.5,.5,.5))

])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((.5,.5,.5),(.5,.5,.5))
])


trainset = torchvision.datasets.CIFAR10(
    root = "./data",
    train = True,
    download= False,
    transform= train_transform

)

valset = torchvision.datasets.CIFAR10(
    root = "./data",
    train = True,
    download= False,
    transform= eval_transform

)

testset = torchvision.datasets.CIFAR10(
    root = "./data",
    train = False,
    download= False,
    transform= eval_transform

)




In [22]:
indices = list(range(len(trainset)))
train_ind , val_ind = train_test_split(indices,test_size= 0.1,random_state=42 ,stratify=trainset.targets)

In [25]:
train_subset = Subset(trainset,train_ind)
val_subset = Subset(valset,val_ind)

In [26]:
train_loader = DataLoader(
    train_subset,
    batch_size= 256,
    shuffle= True,
    num_workers=2,
    pin_memory= True

)

val_loader = DataLoader(
    val_subset,
    batch_size= 256,
    shuffle= False,
    num_workers=2,
    pin_memory= True

)

testset_loader = DataLoader(
    testset,
    batch_size= 256,
    shuffle= False,
    num_workers=2,
    pin_memory= True

)

In [27]:
wideresnet28 = WideResNet28(3,2,28,10)
resnext29 = ResNext29(3,10,29,2)
dualbranchresnet28 = DualBranchResNet28(3,28,10)
kasranet_v1_incepNeXt = KasraNet_V1_IncepNeXt(3,10,28,16,2)



wideresnet28 = wideresnet28.to(device, memory_format=torch.channels_last)
resnext29 = resnext29.to(device, memory_format=torch.channels_last)
dualbranchresnet28 = dualbranchresnet28.to(device, memory_format=torch.channels_last)
kasranet_v1_incepNeXt = kasranet_v1_incepNeXt.to(device, memory_format=torch.channels_last)

In [28]:
EPOCHS = 150
optimizer_wr28 = torch.optim.SGD(wideresnet28.parameters(),lr=0.1,momentum=0.9,weight_decay=5e-4)
optimizer_rn29 = torch.optim.SGD(resnext29.parameters(),lr=0.1,momentum=0.9,weight_decay=5e-4)
optimizer_dr28 = torch.optim.SGD(dualbranchresnet28.parameters(),lr=0.1,momentum=0.9,weight_decay=5e-4)
optimizer_kasrav1 = torch.optim.SGD(kasranet_v1_incepNeXt.parameters(),lr=0.1,momentum=0.9,weight_decay=5e-4)

scheduler_wr28 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_wr28, T_max=EPOCHS, eta_min=1e-6)
scheduler_rn29 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_rn29, T_max=EPOCHS, eta_min=1e-6)
scheduler_dr28 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_dr28, T_max=EPOCHS, eta_min=1e-6)
scheduler_kasrav1 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_kasrav1, T_max=EPOCHS, eta_min=1e-6)

In [29]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

In [30]:
train_wr28 =  Trainer(wideresnet28,optimizer_wr28,criterion,device,scheduler_wr28)
train_rn29 =  Trainer(resnext29,optimizer_rn29,criterion,device,scheduler_rn29)
train_dr28 =  Trainer(dualbranchresnet28,optimizer_dr28,criterion,device,scheduler_dr28)
train_kv1 = Trainer(kasranet_v1_incepNeXt,optimizer_kasrav1,criterion,device,scheduler_kasrav1)

In [32]:
!pip install thop

In [33]:
from thop import profile, clever_format
list_of_models = [wideresnet28,resnext29,dualbranchresnet28,kasranet_v1_incepNeXt]
dummy_input = torch.randn(1, 3, 32, 32).to(device)

for model in list_of_models:
    model.eval() # بهتر است مدل در حالت ارزیابی باشد
    
    # محاسبه MACs (ضرب و جمع) و تعداد پارامترها
    macs, params = profile(model, inputs=(dummy_input, ), verbose=False)
    
    # در ادبیات بینایی ماشین معمولاً هر 1 MAC معادل 2 FLOPs در نظر گرفته می‌شود
    flops = macs * 2 
    
    # فرمت‌دهی خوانا (تبدیل به میلیون و گیگا)
    flops_fmt, macs_fmt, params_fmt = clever_format([flops, macs, params], "%.2f")
    
    print(f"Model: {model.__class__.__name__}")
    print(f"  Parameters: {params_fmt}")
    print(f"  MACs:       {macs_fmt}")
    print(f"  FLOPs:      {flops_fmt}")
    print("-" * 40)

Model: WideResNet28
  Parameters: 1.47M
  MACs:       216.26M
  FLOPs:      432.52M
----------------------------------------
Model: ResNext29
  Parameters: 1.42M
  MACs:       216.88M
  FLOPs:      433.77M
----------------------------------------
Model: DualBranchResNet28
  Parameters: 735.23K
  MACs:       111.19M
  FLOPs:      222.37M
----------------------------------------
Model: KasraNet_V1_IncepNeXt
  Parameters: 1.06M
  MACs:       157.67M
  FLOPs:      315.33M
----------------------------------------


In [14]:
history_wr28=train_wr28.fit(train_loader,val_loader,EPOCHS,"wr28")

Starting training on device: cuda (GPUs: 2, AMP: True) for 150 epochs...

Epoch [01/150] | LR: 0.100000 | Train Loss: 1.8473 - Train Acc: 36.24% | Val Loss: 1.8511 - Val Acc: 43.56% --> (New Best! Saved to ./checkpoints/wr28.pth)
Epoch [02/150] | LR: 0.099989 | Train Loss: 1.4925 - Train Acc: 54.50% | Val Loss: 1.4847 - Val Acc: 57.44% --> (New Best! Saved to ./checkpoints/wr28.pth)
Epoch [03/150] | LR: 0.099956 | Train Loss: 1.2983 - Train Acc: 64.25% | Val Loss: 1.3783 - Val Acc: 62.70% --> (New Best! Saved to ./checkpoints/wr28.pth)
Epoch [04/150] | LR: 0.099901 | Train Loss: 1.1796 - Train Acc: 70.00% | Val Loss: 1.2324 - Val Acc: 68.94% --> (New Best! Saved to ./checkpoints/wr28.pth)
Epoch [05/150] | LR: 0.099825 | Train Loss: 1.1038 - Train Acc: 73.78% | Val Loss: 1.2305 - Val Acc: 69.18% --> (New Best! Saved to ./checkpoints/wr28.pth)
Epoch [06/150] | LR: 0.099726 | Train Loss: 1.0573 - Train Acc: 75.56% | Val Loss: 1.1956 - Val Acc: 71.24% --> (New Best! Saved to ./checkpoints/

In [34]:
history_rn29=train_rn29.fit(train_loader,val_loader,EPOCHS,"rn29")

Starting training on device: cuda (GPUs: 2, AMP: True) for 150 epochs...

Epoch [01/150] | LR: 0.100000 | Train Loss: 1.8262 - Train Acc: 36.72% | Val Loss: 1.7912 - Val Acc: 44.42% --> (New Best! Saved to ./checkpoints/rn29.pth)
Epoch [02/150] | LR: 0.099989 | Train Loss: 1.4587 - Train Acc: 56.21% | Val Loss: 1.7739 - Val Acc: 49.06% --> (New Best! Saved to ./checkpoints/rn29.pth)
Epoch [03/150] | LR: 0.099956 | Train Loss: 1.2960 - Train Acc: 64.58% | Val Loss: 1.3062 - Val Acc: 66.04% --> (New Best! Saved to ./checkpoints/rn29.pth)
Epoch [04/150] | LR: 0.099901 | Train Loss: 1.2111 - Train Acc: 68.84% | Val Loss: 1.2038 - Val Acc: 70.22% --> (New Best! Saved to ./checkpoints/rn29.pth)
Epoch [05/150] | LR: 0.099825 | Train Loss: 1.1470 - Train Acc: 71.98% | Val Loss: 1.1759 - Val Acc: 70.94% --> (New Best! Saved to ./checkpoints/rn29.pth)
Epoch [06/150] | LR: 0.099726 | Train Loss: 1.0966 - Train Acc: 74.72% | Val Loss: 1.4635 - Val Acc: 64.70%
Epoch [07/150] | LR: 0.099606 | Train 

In [35]:
history_dr28=train_dr28.fit(train_loader,val_loader,EPOCHS,"dr28")

Starting training on device: cuda (GPUs: 2, AMP: True) for 150 epochs...

Epoch [01/150] | LR: 0.100000 | Train Loss: 1.8277 - Train Acc: 36.79% | Val Loss: 1.7618 - Val Acc: 43.60% --> (New Best! Saved to ./checkpoints/dr28.pth)
Epoch [02/150] | LR: 0.099989 | Train Loss: 1.4679 - Train Acc: 55.72% | Val Loss: 1.3509 - Val Acc: 61.66% --> (New Best! Saved to ./checkpoints/dr28.pth)
Epoch [03/150] | LR: 0.099956 | Train Loss: 1.2949 - Train Acc: 64.45% | Val Loss: 1.3561 - Val Acc: 63.76% --> (New Best! Saved to ./checkpoints/dr28.pth)
Epoch [04/150] | LR: 0.099901 | Train Loss: 1.1827 - Train Acc: 69.93% | Val Loss: 1.3045 - Val Acc: 66.62% --> (New Best! Saved to ./checkpoints/dr28.pth)
Epoch [05/150] | LR: 0.099825 | Train Loss: 1.1140 - Train Acc: 73.08% | Val Loss: 1.1417 - Val Acc: 72.36% --> (New Best! Saved to ./checkpoints/dr28.pth)
Epoch [06/150] | LR: 0.099726 | Train Loss: 1.0661 - Train Acc: 75.36% | Val Loss: 1.1265 - Val Acc: 73.30% --> (New Best! Saved to ./checkpoints/

In [36]:
history_kv1 = train_kv1.fit(train_loader, val_loader, EPOCHS, "kv1")

Starting training on device: cuda (GPUs: 2, AMP: True) for 150 epochs...

Epoch [01/150] | LR: 0.100000 | Train Loss: 1.8409 - Train Acc: 36.39% | Val Loss: 1.7689 - Val Acc: 45.40% --> (New Best! Saved to ./checkpoints/kv1.pth)
Epoch [02/150] | LR: 0.099989 | Train Loss: 1.4474 - Train Acc: 57.02% | Val Loss: 1.5469 - Val Acc: 57.98% --> (New Best! Saved to ./checkpoints/kv1.pth)
Epoch [03/150] | LR: 0.099956 | Train Loss: 1.2663 - Train Acc: 66.11% | Val Loss: 1.1967 - Val Acc: 69.54% --> (New Best! Saved to ./checkpoints/kv1.pth)
Epoch [04/150] | LR: 0.099901 | Train Loss: 1.1614 - Train Acc: 70.96% | Val Loss: 1.2563 - Val Acc: 67.16%
Epoch [05/150] | LR: 0.099825 | Train Loss: 1.0810 - Train Acc: 74.81% | Val Loss: 1.1837 - Val Acc: 70.80% --> (New Best! Saved to ./checkpoints/kv1.pth)
Epoch [06/150] | LR: 0.099726 | Train Loss: 1.0287 - Train Acc: 77.27% | Val Loss: 1.1259 - Val Acc: 73.42% --> (New Best! Saved to ./checkpoints/kv1.pth)
Epoch [07/150] | LR: 0.099606 | Train Loss:

In [20]:
from IPython.display import FileLink
FileLink("checkpoints/wr28.pth")

/kaggle/working/checkpoints/wr28.pth

In [41]:
import torch
import torch.nn as nn
import pandas as pd

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
eval_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

models_info = [
    ("WideResNet-28-2", WideResNet28(3, 2, 28, 10), "./checkpoints/wr28.pth"),
    ("ResNeXt-29", ResNext29(3, 10, 29, 2), "./checkpoints/rn29.pth"),
    ("DualBranchResNet-28", DualBranchResNet28(3, 28, 10), "./checkpoints/dr28.pth"),
    ("KasraNet-V1-IncepNeXt", KasraNet_V1_IncepNeXt(3, 10, 28, 16, 2), "./checkpoints/kv1.pth"),
]

def measure_gpu_latency(model, batch_size=256, warmup=30, runs=100):
    dummy = torch.randn(batch_size, 3, 32, 32, device=device).to(memory_format=torch.channels_last)
    starter = torch.cuda.Event(enable_timing=True)
    ender = torch.cuda.Event(enable_timing=True)

    with torch.inference_mode(), torch.amp.autocast("cuda"):
        for _ in range(warmup):
            _ = model(dummy)
        torch.cuda.synchronize()

        starter.record()
        for _ in range(runs):
            _ = model(dummy)
        ender.record()
        torch.cuda.synchronize()

    avg_batch_ms = starter.elapsed_time(ender) / runs
    throughput = (batch_size / avg_batch_ms) * 1000.0
    return avg_batch_ms, throughput

results = []

for name, model, ckpt_path in models_info:
    ckpt = torch.load(ckpt_path, map_location=device)
    state_dict = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
    # حذف بافرهای اضافی thop از فایل وزن ذخیره‌شده
    state_dict = {k: v for k, v in state_dict.items() if not k.endswith(("total_ops", "total_params"))}

    model.load_state_dict(state_dict)
    model.to(device, memory_format=torch.channels_last)
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    with torch.inference_mode(), torch.amp.autocast("cuda"):
        for images, labels in testset_loader:
            images = images.to(device, non_blocking=True, memory_format=torch.channels_last)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = eval_criterion(outputs, labels)

            total_loss += loss.item() * labels.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    test_loss = total_loss / total
    test_acc = 100.0 * correct / total
    params_m = sum(p.numel() for p in model.parameters()) / 1e6
    batch_ms, img_per_sec = measure_gpu_latency(model, batch_size=256)

    results.append({
        "Model": name,
        "Params (M)": round(params_m, 2),
        "Test Loss": round(test_loss, 4),
        "Test Acc (%)": round(test_acc, 2),
        "Batch Latency (ms)": round(batch_ms, 2),
        "Throughput (img/s)": int(img_per_sec),
    })

df_results = pd.DataFrame(results).sort_values(by="Test Acc (%)", ascending=False)
print(df_results.to_string(index=False))

                Model  Params (M)  Test Loss  Test Acc (%)  Batch Latency (ms)  Throughput (img/s)
KasraNet-V1-IncepNeXt        1.06     0.6166         95.27               71.60                3575
           ResNeXt-29        1.42     0.6324         95.15               22.32               11467
      WideResNet-28-2        1.47     0.6397         94.45               13.56               18879
  DualBranchResNet-28        0.74     0.6399         94.34               12.40               20638


In [38]:
from IPython.display import display, FileLink

display(FileLink("checkpoints/rn29.pth"))
display(FileLink("checkpoints/dr28.pth"))
display(FileLink("checkpoints/kv1.pth"))

/kaggle/working/checkpoints/rn29.pth

/kaggle/working/checkpoints/dr28.pth

/kaggle/working/checkpoints/kv1.pth